# Phase 0 — DML pricer on Colab Pro+

Trains the Differential-ML option pricer with maturity-gated variance:

  - Backbone: Black-Scholes closed form
  - Residual: τ-gated tanh correction learned from data
  - AAD Greeks (Δ, Γ, 𝒱) in one forward pass

**Gate to pass**: ATM τ=1d Greek error ≤ 2% on all four outputs.

**Runtime**: ~5-10 min on T4/A100. On H100 Colab, ~2 min.

**Before running**: *Runtime → Change runtime type → GPU*. T4 or better is fine here — DML is small.

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, sys
print('python:', sys.version.split()[0], 'torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

## 2. Mount Drive (checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tradefm_ckpts
!ls /content/drive/MyDrive/tradefm_ckpts

## 3. Clone repo + install

In [ ]:
REPO_URL = 'https://github.com/YOURUSER/market-pattern-bot'  # <-- edit
import os
if not os.path.exists('/content/market-pattern-bot'):
    !git clone --depth=1 {REPO_URL} /content/market-pattern-bot
os.chdir('/content/market-pattern-bot')
!pip install -q -r requirements.txt
!pip install -q pyarrow matplotlib

## 4. Train DML — BS pretrain + Heston fine-tune

Writes `checkpoints/dml_pricer.pt` to Drive when done.

In [ ]:
import sys
sys.path.insert(0, '/content/market-pattern-bot')

import torch
from models.config import DMLConfig
from odte.dml_pricer import DMLPricer, greek_error_on_atm
from odte.train.train_dml import pretrain_bs, finetune_heston

device = 'cuda'
pricer = DMLPricer(DMLConfig()).to(device)

# BS pretrain
pretrain_bs(pricer, steps=3000, batch=1024, device=device)
err_bs = greek_error_on_atm(pricer, device=device)
print('after BS pretrain:', err_bs)

# Heston fine-tune
finetune_heston(pricer, steps=600, batch=64, device=device,
                n_mc_paths=2000, n_mc_steps=32, lr=1e-4)
err_hs = greek_error_on_atm(pricer, device=device)
print('after Heston fine-tune:', err_hs)

# Save
out = '/content/drive/MyDrive/tradefm_ckpts/dml_pricer.pt'
torch.save({'state': pricer.state_dict(),
            'cfg': DMLConfig().__dict__,
            'greek_err': err_hs}, out)
print('saved →', out)

## 5. Gate check

In [ ]:
GATE_PCT = 2.0
worst = max(err_hs.values())
print(f'max Greek err = {worst:.3f}%  (gate = {GATE_PCT}%)')
if worst <= GATE_PCT:
    print('\n✅ PASS — DML pricer ready for Phase-1 TradeFM run.')
else:
    print('\n❌ FAIL — increase pretrain_steps or tune DMLConfig.')
    print('   Look at the training loss curve and check for τ-gate tuning issues.')

## 6. Plot Greek error heat-map (diagnostic)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch

pricer.eval()
S_ref = 5500.0
Ks = np.linspace(0.90, 1.10, 21) * S_ref
taus = np.linspace(1/(365*24), 30/365, 25)   # 1h .. 30d
err = np.zeros((len(taus), len(Ks)))

for i, tau in enumerate(taus):
    for j, K in enumerate(Ks):
        S_t = torch.tensor([S_ref], device=device, dtype=torch.float32)
        K_t = torch.tensor([K],     device=device, dtype=torch.float32)
        tau_t = torch.tensor([tau], device=device, dtype=torch.float32)
        r_t = torch.tensor([0.0],   device=device, dtype=torch.float32)
        sig_t = torch.tensor([0.2], device=device, dtype=torch.float32)
        p_h, d_h, g_h, v_h = pricer(S_t, K_t, tau_t, r_t, sig_t)
        from odte.dml_pricer import bs_price_call, bs_greeks_call
        p_t = bs_price_call(S_t, K_t, tau_t, sig_t, r_t)
        d_t, g_t, v_t = bs_greeks_call(S_t, K_t, tau_t, sig_t, r_t)
        err[i, j] = abs(float(d_h.item()) - float(d_t.item())) * 100

plt.figure(figsize=(8, 4.5))
plt.imshow(err, aspect='auto', origin='lower',
           extent=[Ks.min(), Ks.max(), taus.min()*365, taus.max()*365])
plt.colorbar(label='Delta error %')
plt.xlabel('Strike'); plt.ylabel('τ (days)')
plt.title('DML Delta error (%) vs moneyness × maturity')
plt.tight_layout(); plt.show()